In [23]:
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision import datasets
from PIL import Image


In [ ]:
transform = transforms.Compose([
transforms.Resize((256,256)),
transforms.ToTensor()
])

dataTrain = datasets.ImageFolder(root = r"C:\Users\badis\Desktop\cat vs dog classification\archive\dogcat\train",transform = transform)


class DataTest(Dataset):
    def __init__(self,folderpath,transform):
        self.folder_path = folderpath
        self.file_list = os.listdir(self.folder_path)
        self.transform = transform

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self,index):
        filename = self.file_list[index] 
        full_path = os.path.join(self.folder_path, filename)
        img = Image.open(full_path)
        img = img.convert('RGB')
        return self.transform(img)

dataTest = DataTest(r"C:\Users\badis\Desktop\cat vs dog classification\archive\dogcat\test1\test1", transform)





12500
torch.Size([3, 256, 256])


In [34]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.ConvLayer = nn.Conv2d(in_channels = 3, out_channels = 12, kernel_size = 3)
        self.MaxPool = nn.MaxPool2d(kernel_size = 4)
        self.FirstLayer = nn.Linear(47628,6804)
        self.SecondLayer = nn.Linear(6804,567)
        self.ThirdLayer = nn.Linear(567,1)

    def forward(self,data):
        relu = nn.ReLU()
        first = self.ConvLayer(data)
        first = relu(first)
        second = self.MaxPool(first)
        second = torch.flatten(second, start_dim = 1, end_dim = 3)
        third = self.FirstLayer(second)
        third = relu(third)
        fourth = self.SecondLayer(third)
        fourth = relu(fourth)
        fifth = self.ThirdLayer(fourth)
        sigmoid = nn.Sigmoid()
        return sigmoid(fifth)
        



In [47]:
Loss = nn.BCELoss()
NeuralNetwork = Model()
optimizer = torch.optim.SGD(NeuralNetwork.parameters(), lr = 0.1)

DataTrainOBJ = DataLoader(dataTrain, batch_size = 32, shuffle = True)

for i in range(50):
    for batch in DataTrainOBJ:
        image, label = batch
        prediction = NeuralNetwork(image)
        prediction = prediction.squeeze(dim = 1)
        label = label.float()
        prediction = prediction.float()
        results = Loss(prediction, label)
        results.backward()
        optimizer.step()
        optimizer.zero_grad()
        



KeyboardInterrupt: 

In [ ]:
dataValid = datasets.ImageFolder(root = r"C:\Users\badis\Desktop\cat vs dog classification\archive\dogcat\validation", transform = transform)
DataValidOBJ = DataLoader(dataValid, batch_size = 32, shuffle = True)
total = 0
for batch in DataValidOBJ:
    images, labels = batch
    prediction = NeuralNetwork(images)
    prediction = prediction.squeeze(dim = 1)
    prediction = (prediction > 0.5).float()
    ans = (prediction == labels)
    total += ans.sum()

accuracy = total / len(dataValid) * 100

print(accuracy)